# 12_LLM_Fallback.ipynb
====================
This notebook demonstrates the **SupportAI Decision Engine**, a 3-tier customer support ticket routing architecture:
1. **High Confidence (Auto-Route)**: If intent classification confidence > high_threshold, return predicted category.
2. **Mid Confidence (LLM Fallback)**: If confidence is between low and high threshold, perform semantic search on historical resolved cases, construct a contextual instructions prompt, and query a cached small LLM (e.g., Phi-3 Mini) to generate a draft reply.
3. **Low Confidence (Human Escalation)**: If confidence < low_threshold, route directly to human customer support agents.

To support CPU execution on low-memory machines (>2GB RAM), the backend is configured to use a lightweight model.

In [ ]:
from pathlib import Path
import json

from src.models.transformer.decision_engine import DecisionEngine
from src.utils.constants import OUTPUT_DIR

## 1. Define Engine Configuration
We specify the thresholds and model identifiers, defaulting to a lightweight, resource-friendly model for CPU inference.

In [ ]:
config = {
    "decision_engine": {
        "high_confidence_threshold": 0.90,
        "low_confidence_threshold": 0.60
    },
    "llm": {
        "enabled": True,
        "provider": "huggingface",
        "model_id": "hf-internal-testing/tiny-random-gpt2", # Or microsoft/Phi-3-mini-4k-instruct / Qwen/Qwen2.5-0.5B-Instruct
        "max_new_tokens": 64,
        "temperature": 0.2
    }
}

model_dir = OUTPUT_DIR / "models" / "best_model"
retriever_index_dir = OUTPUT_DIR / "retrieval_index"

print("Configuring Decision Engine...")

## 2. Initialise and Cache Decision Engine
We load all model components (intent classifier, FAISS index, SentenceTransformer, and small LLM) into memory once to minimise request latency.

In [ ]:
engine = DecisionEngine(
    model_dir=model_dir,
    retriever_index_dir=retriever_index_dir,
    config=config
)
print("Decision Engine successfully loaded and cached in memory.")

## 3. Tier 1: Auto Route (High Confidence)
We route a ticket where classifier confidence is high.

In [ ]:
# Set threshold low to force auto-routing demonstration
engine.high_threshold = 0.01
res_high = engine.route_ticket("I forgot my passcode and cannot login")
print(json.dumps(res_high, indent=2))

## 4. Tier 2: LLM Fallback (Mid Confidence)
If confidence is within the medium range, the engine fetches top-3 similar resolved cases and drafts a reply using the LLM.

In [ ]:
# Set high threshold high and low threshold low to trigger fallback demonstration
engine.high_threshold = 0.99
engine.low_threshold = 0.01
res_mid = engine.route_ticket("reset card pin number")
print(json.dumps(res_mid, indent=2))

## 5. Tier 3: Human Escalation (Low Confidence)
If confidence is extremely low, the ticket is routed directly to a human specialist, saving LLM resource costs.

In [ ]:
# Set thresholds high to force human escalation route
engine.high_threshold = 0.99
engine.low_threshold = 0.95
res_low = engine.route_ticket("random garbled input query")
print(json.dumps(res_low, indent=2))